> **Synthetic data.** This notebook reads seeded mock tables in `data/synthetic/` (filenames start with `SYNTHETIC_`). They are not MIMIC-III and contain no real patient records. See `DATA.md`.


In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:80% !important; }</style>"))

In [ ]:
import math
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

In [ ]:
import sys
from pathlib import Path
for _root in (Path.cwd(), Path.cwd().parent):
    if (_root / "inflammation_risk_models").is_dir():
        sys.path.insert(0, str(_root.resolve()))
        break
Path('../images').mkdir(parents=True, exist_ok=True)


In [ ]:
from inflammation_risk_models.visualization.plots import duplicated_ids_barplot, duplicated_ids_barplot, plot_inflammation_controls_prop_pie_chart, shared_countplot, plot_pie_chart, plot_missing_values_pie_chart, plot_multiple_boxplot, shared_histplot, remove_outliers, correlation_matrix_heatmap
from inflammation_risk_models.visualization.util import *


In [ ]:
static_variables_df = pd.read_csv('../data/synthetic/SYNTHETIC_static_variables.csv')
controls_static_variables_df = pd.read_csv('../data/synthetic/SYNTHETIC_static_variables_controls.csv')
cases_static_variables_df = pd.read_csv('../data/synthetic/SYNTHETIC_static_variables_cases.csv')


Static data are available for both control patients and inflammation cases. An exploratory analysis is then carried out on their demographic data and administrative information associated with their stay in the ICU.


The data from each group will be studied independently using the same exploratory methods. Subsequently, the complete dataset will be used to analyze the entire population and to perform comparisons.

### Entire population (control cases and inflammation cases)

In [ ]:
static_variables_df.head()

The complete population dataset includes the following fields:”

subject_id: Patient ID

hadm_id: Hospital admission ID

icustay_id: ICU stay ID

gender: Gender

admittime: Time of hospital admission

admission_age: Age at hospital admission

ethnicity: Ethnicity

admission_type: Reason for hospital admission

admission_location: Source of hospital admission

first_careunit: Type of ICU in the first admission

hospstay_seq: Hospital admission sequence number

first_hosp_stay: First hospital stay indicator

intime: ICU admission time (current hospitalization)

outtime: ICU discharge time (current hospitalization)

icustay_seq: ICU admission sequence number

first_icu_stay: First ICU admission indicator

In [ ]:
static_variables_df.info()

In [ ]:
static_variables_df['admittime'] = pd.to_datetime(static_variables_df['admittime'])
static_variables_df['intime'] = pd.to_datetime(static_variables_df['intime'])
static_variables_df['outtime'] = pd.to_datetime(static_variables_df['outtime'])
static_variables_df['length_of_stay'] = static_variables_df['outtime'] - static_variables_df['intime']
static_variables_df['length_of_stay'] = static_variables_df['length_of_stay']/ pd.to_timedelta(1, unit='h')

### Control group

In [ ]:
controls_static_variables_df = controls_static_variables_df.drop(columns=['hadm_id.1', 'icustay_id.1', 'intime.1', 'outtime.1'])
controls_static_variables_df.columns.difference(static_variables_df.columns)

In [ ]:
controls_static_variables_df.head()

“The set of patients classified as control cases includes the following additional fields:”

control_onset_hour: inflammation onset hour (not real)

control_onset_time: inflammation onset time (not real)

length_of_stay: Duration of ICU stay

matched_case_icustay_id

In [ ]:
controls_static_variables_df.info()

In [ ]:
controls_static_variables_df['admittime'] = pd.to_datetime(controls_static_variables_df['admittime'])
controls_static_variables_df['intime'] = pd.to_datetime(controls_static_variables_df['intime'])
controls_static_variables_df['outtime'] = pd.to_datetime(controls_static_variables_df['outtime'])

### Casos de inflamation

In [ ]:
cases_static_variables_df = cases_static_variables_df.drop(columns=['icustay_id.1', 'intime.1', 'outtime.1'])
cases_static_variables_df.columns.difference(static_variables_df.columns)

In [ ]:
cases_static_variables_df.head()

El conjunto de pacientes que son casos de inflammation cuentan con los siguientes campos adicionales:
- delta_score: Incremento del valor del SOFA
- length_of_stay: Duración de la estancia en la UCI
- inflammation_onset: Momento del comienzo de la inflammation
- inflammation_onset_day: Día del comienzo de la inflammation
- inflammation_onset_hour: Hora del comienzo de la inflammation

In [ ]:
cases_static_variables_df.info()

In [ ]:
cases_static_variables_df['admittime'] = pd.to_datetime(cases_static_variables_df['admittime'])
cases_static_variables_df['intime'] = pd.to_datetime(cases_static_variables_df['intime'])
cases_static_variables_df['outtime'] = pd.to_datetime(cases_static_variables_df['outtime'])

-------------------

Debemos comprobar la unicidad de las entradas de los conjuntos de datos ya que las entradas se componen de un ID asociado al paciente, un ID asociado a la admisión al hospital y un ID asociado a la estancia en la UCI.

In [ ]:
static_variables_ids_duplicates = get_duplicates_count_ids(static_variables_df)
static_variables_ids_duplicates

In [ ]:
duplicated_ids_barplot(static_variables_ids_duplicates, 'Unique entries by ID')

In [ ]:
controls_static_variables_ids_duplicates = get_duplicates_count_ids(controls_static_variables_df)
controls_static_variables_ids_duplicates

In [ ]:
duplicated_ids_barplot(controls_static_variables_ids_duplicates, 'Unique entries by ID')

In [ ]:
cases_static_variables_ids_duplicates = get_duplicates_count_ids(cases_static_variables_df)
cases_static_variables_ids_duplicates

In [ ]:
duplicated_ids_barplot(cases_static_variables_ids_duplicates, 'Unique entries by ID')

Podemos ver como para los tres posibles IDs no existen múltiples entradas, mientras que para el ID del sujeto y el ID de admisión al hospital sí. Esto se debe a que los pacientes pueden ser hospitalizados en diferentes unidades de la UCI durante la misma estancia en el hospital.

Es consecuencia directa de este hecho que existan entradas duplicadas en base al ID del paciente o al ID de la admisión al hospital.

Extraemos los IDs de los pacientes que pertenecen al grupo de casos de control y casos de inflammation respectivamente.

In [ ]:
controls_subjects_ids = controls_static_variables_df['subject_id'].unique()
cases_subjects_ids = cases_static_variables_df['subject_id'].unique()

## Casos de inflammation y control

In [ ]:
plot_inflammation_controls_prop_pie_chart(controls_static_variables_df, cases_static_variables_df, save_path='../images/proporciones_inflammation_control.png')

## Género

Para la graficación del recuento de pacientes de distintos géneros debemos tomar aquellas entradas que sean únicas en base al ID del sujeto, ya que se trata de un campo que no se ve afectado por el resto de columnas.

In [ ]:
static_variables_unique_subjects = static_variables_df.drop_duplicates(subset=['subject_id'])
static_variables_unique_subjects = static_variables_unique_subjects.reset_index(drop=True)

controls_static_variables_unique_subjects = controls_static_variables_df.drop_duplicates(subset=['subject_id'])
controls_static_variables_unique_subjects = controls_static_variables_unique_subjects.reset_index(drop=True)

cases_static_variables_unique_subjects = cases_static_variables_df.drop_duplicates(subset=['subject_id'])
cases_static_variables_unique_subjects = cases_static_variables_unique_subjects.reset_index(drop=True)

In [ ]:
control_cases_subject_ids = pd.concat([controls_static_variables_unique_subjects['subject_id'], cases_static_variables_unique_subjects['subject_id']])
control_cases_subject_ids = control_cases_subject_ids.drop_duplicates()
control_cases_subject_ids = control_cases_subject_ids.reset_index(drop=True)

In [ ]:
# Todos los ids de la union son parte del dataset completo
control_cases_subject_ids.isin(static_variables_unique_subjects['subject_id']).sum() == len(control_cases_subject_ids.isin(static_variables_unique_subjects['subject_id']))

In [ ]:
shared_subjects = []
shared_indexes = []
exclusive_subjects = []
exclusive_indexes = []

for index, row in static_variables_unique_subjects.iterrows():
    if row['subject_id'] in control_cases_subject_ids:
        shared_subjects.append(row['subject_id'])
        shared_indexes.append(index)
    else:
        exclusive_subjects.append(row['subject_id'])
        exclusive_indexes.append(index)

In [ ]:
shared_countplot('gender', static_variables_unique_subjects.loc[exclusive_indexes], 'Control/inflammation cases', controls_static_variables_unique_subjects, 'Control cases', cases_static_variables_unique_subjects, 'inflammation cases')

## Edad y momento de admisión del individuo

Para salvaguardar la privacidad de los pacientes, el campo del momento de admisión de los mismos al igual que otros con datos temporales han sido alterados. Es por ello que no se puede realizar una exploración adecuada de los mismos ni se debe tratar de devolverlos a su estado original para ello.

In [ ]:
import matplotlib.dates as mdates
_intime = pd.to_datetime(static_variables_df['intime']).dropna()
plt.hist(mdates.date2num(_intime.dt.to_pydatetime()))
plt.show()


In [ ]:
plt.hist(static_variables_df['admission_age'])
plt.show()

## Etnia

En cuanto a la etnia de los pacientes, encontramos que existen multitud de etiquetas dentro de las diferentes etnias con muy pocas observaciones.

In [ ]:
static_variables_df['ethnicity'].value_counts()

Con el fin de simplificar el análisis de los datos, se agruparán los distintas etiquetas más precisas dentro de una misma etnia.

In [ ]:
ethnic_subgroups_aggregated = static_variables_unique_subjects.loc[exclusive_indexes]['ethnicity'].copy()

In [ ]:
for index, value in ethnic_subgroups_aggregated.items():
    ethnicity = ethnic_subgroups_aggregated.loc[index]
    
    if '-' in ethnicity:
        ethnic_subgroups_aggregated.loc[index] = ethnicity[:ethnicity.index('-')].strip()


In [ ]:
plot_pie_chart(ethnic_subgroups_aggregated, '', savePath='../images/percentage_ethnicity.png')

## Tipo de admisión

In [ ]:
static_variables_unique_subjects['admission_type'].unique()

In [ ]:
plot_pie_chart(static_variables_unique_subjects['admission_type'], '', savePath='../images/admission_type.png')

## Lugar de admisión

In [ ]:
static_variables_unique_subjects['admission_location'].value_counts()

In [ ]:
plot_pie_chart(static_variables_unique_subjects['admission_location'], '', savePath='../images/admission_location.png')

## Primera unidad de UCI en la que fué admitido

In [ ]:
static_variables_df['first_careunit'].value_counts()

In [ ]:
plot_pie_chart(static_variables_df['first_careunit'], '', savePath='../images/first_careunit.png')

- MICU: Medical Intensive Care Unit.
- CSRU: Cardiac Surgery Recovery Unit.
- SICU: Surgical Intensive Care Unit.
- NICU: Neonatal Intensive Care Unit.
- CCU: Critical Care Unit.
- TSICU: Trauma Surgical Intesive Care Unit.

## Duración (en horas) de la estancia de los pacientes

Para este apartado hemos calculado el tiempo total en horas por cada estancia de cada paciente. Los casos en los que los pacientes seguían ingresado en el momento de toma de datos se ignoran.

In [ ]:
plot_missing_values_pie_chart(static_variables_df, 'length_of_stay', 'tst')
plot_missing_values_pie_chart(controls_static_variables_df, 'length_of_stay', 'tst')
plot_missing_values_pie_chart(cases_static_variables_df, 'length_of_stay', 'tst')

Se eliminan las entradas para las que los pacientes seguían ingresados en el momento de la recolección de los datos

In [ ]:
finished_stay_length_df = static_variables_df.dropna(subset=['length_of_stay'])
controls_finished_stay_length_df = controls_static_variables_df.dropna(subset=['length_of_stay'])
cases_finished_stay_length_df = cases_static_variables_df.dropna(subset=['length_of_stay'])

In [ ]:
print('FULL DATASET')
print(finished_stay_length_df['length_of_stay'].describe())
print()

print('CONTROLS')
print(controls_finished_stay_length_df['length_of_stay'].describe())
print()

print('CASES')
print(cases_finished_stay_length_df['length_of_stay'].describe())
print()

In [ ]:
finished_stay_length_title = 'Full dataset'
controls_finished_stay_length_title = 'Controls'
cases_finished_stay_length_title = 'Cases'

plot_multiple_boxplot('length_of_stay', finished_stay_length_df, finished_stay_length_title, controls_finished_stay_length_df, controls_finished_stay_length_title, cases_finished_stay_length_df, cases_finished_stay_length_title, orient='v', subplots_orientation='h')

In [ ]:
plot_multiple_boxplot('length_of_stay', finished_stay_length_df, 'Full dataset', orient='h', save_path='../images/length_stay_all_boxplot.png')

In [ ]:
plot_multiple_boxplot('length_of_stay', controls_finished_stay_length_df, controls_finished_stay_length_title, cases_finished_stay_length_df, cases_finished_stay_length_title, orient='h', save_path='../images/length_stay_boxplot_controls_cases.png')

In [ ]:
sns.histplot(finished_stay_length_df, x='length_of_stay').set(title='')
plt.show()

In [ ]:
sns.histplot(controls_finished_stay_length_df, x='length_of_stay', kde=True).set(title='')
plt.show()

In [ ]:
sns.histplot(cases_finished_stay_length_df, x='length_of_stay', kde=True).set(title='')
plt.show()

Se puede apreciar que la duración de la estancia de los distintos pacientes cuenta con muchos valores atípicos, por lo que reduciremos su cantidad mediante el z-score.

In [ ]:
finished_stay_length_no_outliers_df = remove_outliers(finished_stay_length_df, 'length_of_stay')
controls_finished_stay_length_no_outliers_df = remove_outliers(controls_finished_stay_length_df, 'length_of_stay')
cases_finished_stay_length_no_outliers_df = remove_outliers(cases_finished_stay_length_df, 'length_of_stay')

In [ ]:
plot_multiple_boxplot('length_of_stay', finished_stay_length_no_outliers_df, finished_stay_length_title, controls_finished_stay_length_no_outliers_df, controls_finished_stay_length_title, cases_finished_stay_length_no_outliers_df, cases_finished_stay_length_title, subplots_orientation='h')

In [ ]:
print('FULL DATASET')
print(finished_stay_length_no_outliers_df['length_of_stay'].describe())
print()

print('CONTROLS')
print(controls_finished_stay_length_no_outliers_df['length_of_stay'].describe())
print()

print('CASES')
print(cases_finished_stay_length_no_outliers_df['length_of_stay'].describe())
print()

In [ ]:
shared_histplot('length_of_stay', finished_stay_length_no_outliers_df, finished_stay_length_title, controls_finished_stay_length_no_outliers_df, controls_finished_stay_length_title, cases_finished_stay_length_no_outliers_df, cases_finished_stay_length_title)

In [ ]:
sns.histplot(finished_stay_length_no_outliers_df, x='length_of_stay', kde=True).set(title='')
plt.show()

In [ ]:
sns.histplot(controls_finished_stay_length_no_outliers_df, x='length_of_stay', kde=True).set(title='')
plt.show()

In [ ]:
sns.histplot(cases_finished_stay_length_no_outliers_df, kde=True, x='length_of_stay')
plt.show()

Para cada uno de los conjuntos de datos se puede apreciar que sus histogramas son sesgados a la derecha, es decir, siguen una distribución dónde un gran número de valores se encuentran en el lado inferior del eje x y la cola a la derecha (que además en este caso se muestra de gran longitud).

Se puede apreciar de manera más clara realizando un KDE (kernel density estimation), mediante el cuál podemos tener una estimación de la función de densidad de probabilidad de la variable aleatoria length_of_stay.

## Primera estancia hospitalizado

In [ ]:
plot_pie_chart(static_variables_df['first_hosp_stay'].map({1:'Primera hospitalización', 0:'Más de una hospitalización'}), 'Porcentaje de primeras hospitalizaciones', savePath='../images/percent_first_hosp.png')

In [ ]:
plot_pie_chart(controls_static_variables_df['first_hosp_stay'].map({1:'Primera hospitalización', 0:'Más de una hospitalización'}), 'Porcentaje de primeras hospitalizaciones')

In [ ]:
plot_pie_chart(cases_static_variables_df['first_hosp_stay'].map({1:'Primera hospitalización', 0:'Más de una hospitalización'}), 'Porcentaje de primeras hospitalizaciones')

## Primera estancia en la UCI

In [ ]:
plot_pie_chart(static_variables_df['first_icu_stay'].map({1:'Primera admisión a la UCI', 0:'Más de una admisión'}), 'Porcentaje de primeras admisiones a la UCI', savePath='../images/percent_fist_icu.png')

In [ ]:
plot_pie_chart(controls_static_variables_df['first_icu_stay'].map({1:'Primera admisión a la UCI', 0:'Más de una admisión'}), 'Porcentaje de primeras admisiones a la UCI')

In [ ]:
plot_pie_chart(cases_static_variables_df['first_icu_stay'].map({1:'Primera admisión a la UCI', 0:'Más de una admisión'}), 'Porcentaje de primeras admisiones a la UCI')

## Control onset

Se define el control onset como el tiempo absoluto desde el momento de admisión (en horas) hasta que el caso emparejado cumple el criterio de inflammation.

In [ ]:
sns.histplot(controls_static_variables_df, kde=True, x='control_onset_hour')
plt.show()

Eliminando los valores atípicos

In [ ]:
sns.histplot(remove_outliers(controls_static_variables_df, 'control_onset_hour'), kde=True, x='control_onset_hour')
plt.show()

## inflammation onset

In [ ]:
sns.histplot(cases_static_variables_df, kde=True, x='inflammation_onset_hour').set_title('')
plt.savefig('../images/dist_inflammation_hour.png', bbox_inches='tight')
plt.show()

In [ ]:
cases_static_variables_df['inflammation_onset_hour'].describe()

Igual que en el conjunto de control, eliminaremos los valores atípicos para poder visualizar de mejor forma la distribución de los valores de inflammation onset.

In [ ]:
sns.histplot(remove_outliers(cases_static_variables_df, 'inflammation_onset_hour'), kde=True, x='inflammation_onset_hour')
plt.show()

Se puede ver como para ambos conjuntos los datos con respecto al comienzo de la inflammation (siendo en los casos de control datos imputados para utilizarse por el modelo y no datos reales) siguen la misma distribución. Esto se debe a la forma que se han tenido de imputarlos en los casos de control ya que toman como momento de comienzo de la inflammation el del caso de inflammation emparejado.

Este enfoque trata de evitar que el modelo utilice datos muy sesgados y trivialice la tarea de predicción basándose mayoritariamente en que este campo exista o no para los distintos casos en lugar de considerar más el resto de valores. Esto se ve incluso más acentuado al tratarse de un conjunto de datos muy desbalanceado, hecho que se trata de evitar con esta técnica y con el emparejamiento entre casos de inflammation y control.

## Proporción de casos de control y de inflammation

Cada caso está emparejado con 10 controles aleatorios no asignados.

In [ ]:
# Comprobamos que se cumple la relación 1:10
matching_proportion_df = controls_static_variables_df.groupby('matched_case_icustay_id')['icustay_id'].count()
matching_proportion_df

In [ ]:
(matching_proportion_df.values == 10).all()

In [ ]:
# Comprobamos que no están asignados a múltiples casos de inflammation
matching_icustay_id_df = controls_static_variables_df.groupby('matched_case_icustay_id')['icustay_id']
matching_icustay_id_df = matching_icustay_id_df.nunique()
matching_icustay_id_df

In [ ]:
(matching_icustay_id_df.values == 10).all()

## Incremento del SOFA

In [ ]:
cases_static_variables_df['delta_score'].describe()

In [ ]:
cases_static_variables_df['delta_score'].value_counts().sort_index()

In [ ]:
sns.boxplot(cases_static_variables_df['delta_score']).set_title('Aumento máximo del SOFA')
plt.show()

In [ ]:
sns.histplot(cases_static_variables_df, kde=True, x='delta_score', bins=10).set_title('Incremento total del SOFA')
plt.savefig('../images/total_increase_sofa.png', bbox_inches='tight')
plt.show()

In [ ]:
cases_static_variables_df['delta_score'].describe()

Dado que se trata del incremento del SOFA para los distintos pacientes, se interpreta que este se corresponde al total o máximo durante su estancia. El incremento progresivo se podrá apreciar en el análisis exploratorio de las series temporales.

## Secuencia de estancias en el hospital

In [ ]:
static_variables_df['hospstay_seq'].describe()

In [ ]:
sns.kdeplot(static_variables_df['hospstay_seq'], fill=True).set_title('')
plt.show()


In [ ]:
static_variables_df['hospstay_seq'].unique()

- Controls:

In [ ]:
controls_static_variables_df['hospstay_seq'].describe()

In [ ]:
sns.kdeplot(controls_static_variables_df['hospstay_seq'], fill=True).set_title('')
plt.show()


- Cases:

In [ ]:
cases_static_variables_df['hospstay_seq'].describe()

In [ ]:
sns.kdeplot(cases_static_variables_df['hospstay_seq'], fill=True).set_title('')
plt.show()


## Secuencia de estancias en la UCI

In [ ]:
static_variables_df['icustay_seq'].describe()

In [ ]:
sns.kdeplot(static_variables_df['icustay_seq'], fill=True)
plt.show()


In [ ]:
static_variables_df['icustay_seq'].unique()

- Controls:

In [ ]:
controls_static_variables_df['icustay_seq'].describe()

In [ ]:
sns.kdeplot(controls_static_variables_df['icustay_seq'], fill=True).set_title('')
plt.show()


- Cases:

In [ ]:
cases_static_variables_df['icustay_seq'].describe()

In [ ]:
sns.kdeplot(cases_static_variables_df['icustay_seq'], fill=True).set_title('')
plt.show()


Se puede apreciar como la mayoría de valores para ambas variables es uno, valor correspondiente a ser la primera admisión en el hospital o la UCI. Sin embargo, las entradas al hospital para cada paciente son mucho mayores que las de la UCI, hecho razonable debido a que el ingreso en la UCI se debe a un estado de salud grave del paciente y por tanto es menos común.

## Correlación entre las variables estáticas

In [ ]:
static_variables_df.info()

No se analizarán los datos correspondientes a los IDs ni fechas puesto que los primeros no son  de relevancia y los segundos se tendrán en cuenta en el análisis de las series temporales. 

- Correlación entre variables numéricas: (Pearson)

In [ ]:
numerical_features = ['admission_age', 'hospstay_seq', 'first_hosp_stay', 'icustay_seq', 'first_icu_stay', 'length_of_stay']

In [ ]:
sns.pairplot(static_variables_df[numerical_features])
plt.show()

In [ ]:
numeric_features_df = static_variables_df[numerical_features]
correlation_matrix_heatmap((15,12), numeric_features_df.corr(), save_path='../images/numeric_statics_corr.png')

Representamos la recta de regresión asociada a las variables que muestran correlación para estudiarla.

In [ ]:
sns.regplot(x='icustay_seq', y='first_icu_stay', data=static_variables_df)
plt.show()

In [ ]:
sns.regplot(x='hospstay_seq', y='first_hosp_stay', data=static_variables_df)
plt.show()

Ninguna de las variables numéricas muestra correlación entre sí, únicamente entre icustay_seq y first_icu_stay y hospstay_seq y first_hosp_stay con una correlación negativa menor que -0,5. En ambos casos, la primera representa el número de la estancia actual del paciente y la segunda si se trata de la primera estancia del mismo.

La alta correlación negativa se debe a que, por naturaleza, un paciente no puede tomar simultáneamente un valor verdadero para la primera estancia y un número para la estancia actual distinto de 1. Por ello, no se trata de un hecho a estudiar y por tanto no es relevante para el estudio.

- Asociación entre variables categóricas:

Ahora estudiaremos la asociación existente entre las distintas variables categóricas mediante la entropía condicional de cada variable dadas las demás.

In [ ]:
categorical_variables = ['gender', 'ethnicity', 'admission_type', 'admission_location', 'first_careunit']

In [ ]:
categorical_features_association = association_matrix(static_variables_df, categorical_variables)

In [ ]:
correlation_matrix_heatmap((15,12), categorical_features_association, vmin=0)

- Asociación entre variables numéricas y categóricas:

In [ ]:
static_variables_df.info()

In [ ]:
mixed_type_features = numerical_features + categorical_variables
mixed_type_features

In [ ]:
correlation_ratio_matrix = correlation_ratio_matrix(static_variables_df, categorical_variables, numerical_features)

In [ ]:
correlation_matrix_heatmap((15,12), correlation_ratio_matrix, vmin=0)

Podemos apreciar que no existe correlación entre las variables categóricas y numéricas anteriores.